## Difference-in-Differnce Unweighted  

The current estimates are:  

    1. Treated Pre vs. Post = -3.54 (most biased)
       1. Compare treated users before promo vs. after promo  
       2. Assumption: Nothing else changed over time  
       3. The control group contrdicts the assumption as their margin +1.06 during the promo  
       4. Mixes promo effect (demand bump) and time trend (seasonality)  
       5. Underestimates the damage of the promo  
       6. "It confuses time effects with treatment effects"  
    2. Naive Post Difference = -5.61 (cross sectional bias)  
       1. Compares the exposed (treatment) vs. unexposed (control) during the promo ONLY  
       2. Biased as exposed users were already stronger customers before the promo (different pre period metrics)  
       3. Mixes true promo effect and selection bias
       4. "It confuses who we targeted with what the promo did
    3. Manual DiD = -4.60  
       1. Assumes: In the absence of treatment, treated and control WOULD HAVE followed parallel trends  
       2. Assumption is not perfectly true in the raw data  
       3. Why? Treatement probability is correlated with baseline economics, stronger users respond differently to seasonal effects  
       4. DiD corrects for Common Time Effects & Baseline level differences  
       5. Removes "Common Time Shocks & Constant Baseline Differnces"  
       6. It does not automatically remove all selection bias
       7. "It corrects for time, but not for all targeting bias"  

### Treated Pre/Post mixes: Treatment (true effect) + Time  
### Naive Post mixes: Treatment + Selection  
### DiD mixes: Treatment + Imperfect assumption  

An estimator is biased whenever it attributes changes in the outcom to treatment that are actually caused by something else.  

"Something Else" ~ Time trends, Targeting differences, Violated assumption  


5️⃣ Simple Analogy

Imagine two runners:
	•	Runner A always runs 2 seconds faster than Runner B.

That’s baseline difference (selection bias).

Now imagine a headwind slows both runners by 3 seconds.

DiD works because:
	•	Both slowed equally.
	•	The 2-second baseline gap remains constant.

But if Runner A handles wind better and only slows by 1 second:

Now trends are not parallel.

DiD becomes biased.  

6️⃣ Clean One-Line Difference

Selection bias = groups are inherently different.  
DiD = subtracts stable differences over time, assuming trends are parallel.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns 
import statsmodels.formula.api as smf

# Read in synthetic data

df = pd.read_csv("data/raw/holiday_promotion_panel.csv")
df.head(1)

,user_id,date,dow,post,exposed,treated_group,discount_pct,region,device_type,traffic_source,...,list_price,gross_sales_pre_discount,net_revenue,unit_cost,total_cost,margin_dollars,margin_rate,counterfactual_units,counterfactual_revenue,counterfactual_margin_dollars
0,0,2025-11-01,5,0,0,0,0.0,South,Desktop,Email,...,16.856929,16.856929,16.856929,11.017525,11.017525,5.839403,0.34641,1,16.856929,5.839403


In [2]:

promo = df[df['post'] == 1] # df filtered by promo period (post==1)
treated_users = promo[promo['exposed'] == 1]['user_id'].unique() #collect all unique uid of treated

df['ever_exposed'] = df['user_id'].isin(treated_users).astype(int) # new df column with binary flag
# ensures we properly label or identify treated customers in the pre period

### 1. DiD Regression Model without Clustering (Plain OLS)  

Plain OLS: Every row is independent  

Coefficient to look for:  

   
      1. post:ever_exposed = -4.5958 -> DiD impact  
      2. std err = 0.701 -> "wiggle" noise aka random fluctuation  
         1. If you ran this same campaign many times under similar conditions, your estimate would typically bounce around by about +/- $0.70 due to randomness or normal day-to-day customer behavior  
      3. The signal size is much larger than the wiggle size
      4. t-stat = -4.5958 / 0.701 -> -6.5 (the est. effect is 6.5 times larger than typical fluctuating variation)  
      5. p-value = 0.000 statistically significant  
      6. All this combined, large signal, small std err, large t-stat, and small p-value -> -$4.5958 per user-day is economically significant 




In [3]:

model = smf.ols(
    "margin_dollars ~ post + ever_exposed + post:ever_exposed",
    data=df
).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:         margin_dollars   R-squared:                       0.022
Model:                            OLS   Adj. R-squared:                  0.022
Method:                 Least Squares   F-statistic:                     2575.
Date:                Fri, 27 Feb 2026   Prob (F-statistic):               0.00
Time:                        02:22:45   Log-Likelihood:            -1.2385e+06
No. Observations:              350000   AIC:                         2.477e+06
Df Residuals:                  349996   BIC:                         2.477e+06
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             3.3148      0.26

### 2. DiD Regression w/ Clustering (OLS w/ cluster)  

The plain OLS the std err assumes "every row is independent. 


Due to the data being at the user-day level, each user appears many times, user margin likely correlated day to day (behavior stays consistent, margin will be the same for the user from day to day or ask the same person 35 questions, you will get 35 similar answers), so observations within a user are not independent.  
  
  
Clustering tells the model observations within a user may be correlated so please adjust the std errs accordingly

4️⃣ Why This Matters in Panel Data  

You have:  
	•	10,000 users  
	•	35 days per user  
	•	350,000 rows  

Without clustering:  
	•	The model thinks it has 350,000 independent observations.  
	•	That inflates precision.  

With clustering:  
	•	The model recognizes effective sample size is closer to 10,000 users.  
	•	Standard errors become more honest.   

⸻

5️⃣ Simple Analogy

Imagine surveying 10,000 people once vs surveying 10,000 people 35 times each.      

If you treat all 350,000 answers as independent,   
you exaggerate how much new information you have.  

Clustering corrects for that.

In [4]:

model_clustered = smf.ols(
    "margin_dollars ~ post + ever_exposed + post:ever_exposed",
    data=df
).fit(cov_type="cluster", cov_kwds={"groups": df["user_id"]})

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:         margin_dollars   R-squared:                       0.022
Model:                            OLS   Adj. R-squared:                  0.022
Method:                 Least Squares   F-statistic:                     2575.
Date:                Fri, 27 Feb 2026   Prob (F-statistic):               0.00
Time:                        02:22:50   Log-Likelihood:            -1.2385e+06
No. Observations:              350000   AIC:                         2.477e+06
Df Residuals:                  349996   BIC:                         2.477e+06
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             3.3148      0.26

### Two-Way Fixed Effects Regression Model (User + Date FE, clustered SEs)

Control for:  

    1. User Fixed Effects: each user's baseline profitability level  

    2. Date Fixed Effects: day-specific shocks (seasonality, day of week patterns, holiday ramp, etc.)   

       1. Remember: Selection bias existed because treated users were inherently stronger  
       2. User FE: Removes all stable differences in margin levels across users  
       3. TWFE addresses selection bias more aggressively than basic DiD  
       4. It cannot fix time-varying bias, BUT it fixes baseline heterogentity completely  
       5. Basic DiD: Compare treated vs. control trends  
       6. TWFE DiD: Compare each treated user to themselves over time, while removing any daily shocks that effect everyone  

    3. It's a cleaner ISOLATION of the TREATMENT EFFECT

In [5]:
# DUMMY VARIABLE EXPLOSION
# twfe = smf.ols(
#     "margin_dollars ~ post:ever_exposed + C(user_id) + C(date)",
#     data=df
# ).fit(cov_type='cluster', cov_kwds={"groups": df['user_id']})

# print(twfe.summary())

### "within transformation" TWE

In [8]:
import statsmodels.api as sm 

df2 = df.copy()
df2["x"] = df2["post"] * df2["ever_exposed"]  # the DiD interaction

# Means
y = df2["margin_dollars"]
x = df2["x"]

y_i = df2.groupby("user_id")["margin_dollars"].transform("mean")
y_t = df2.groupby("date")["margin_dollars"].transform("mean")
y_bar = y.mean()

x_i = df2.groupby("user_id")["x"].transform("mean")
x_t = df2.groupby("date")["x"].transform("mean")
x_bar = x.mean()

# Two-way demeaning
y_tilde = y - y_i - y_t + y_bar
x_tilde = x - x_i - x_t + x_bar

# OLS on transformed variables (no intercept needed)
model = sm.OLS(y_tilde, x_tilde).fit(cov_type="cluster", cov_kwds={"groups": df2["user_id"]})

print("TWFE coef:", model.params.iloc[0])
print("TWFE se:  ", model.bse.iloc[0])
print("TWFE p:   ", model.pvalues.iloc[0])

TWFE coef: -4.59576355709437
TWFE se:   0.6456584603223602
TWFE p:    1.0954542240839984e-12


2️⃣ What This Means Conceptually  

Adding:  
	•	User fixed effects  
	•	Date fixed effects  

Did not materially change the estimate.  

Why?  

Because your original DiD model already accounted for:  
	•	Baseline group differences (ever_exposed)  
	•	Time shift (post)  

In your DGP, the remaining bias was not coming from time-invariant user heterogeneity.  
**Time-Invariant -> time independent, time is consistent, system remains unchanged over time**  
"Time-invariant user heterogeneity means users have STABLE characteristics that make them CONSISTENTLY DIFFERENT from one another over time."  

		1. User A is a big spender with margin = $8 per day
		2. User C is a low spender with margin = $1 per day  
		3. These differnces are persistent (different users, different margin, stable/consistent)
		4. User A doesn't suddently become User C tomorrow  
		5. Stable Difference = Time-invariant heterogentity  

Time-invariant user heterogeneity = stable differences across users that don’t change over time.  


So user FE didn’t move the needle.  

Clean Comparison  

Time-invariant heterogeneity  
= User A always higher than User B.  
→ Fixed effects remove this.  

Time-varying confounding  
= User A increases faster than User B over time.  
→ Fixed effects do NOT remove this.  

⸻

Why This Explains Your Results  

Your TWFE estimate stayed at -4.6 instead of moving to -6.5.  

That suggests:  

		The remaining bias is not from stable baseline differences.  

		It’s from differences in how groups evolve over time.    


		Time-invariant heterogeneity is about different levels.  
		Time-varying confounding is about different slopes.  

		Fixed effects remove level differences.  
		They do not remove slope differences.

If TWFE moves closer to -6.54, it’s because user FE removed a lot of the stable selection bias.  

If it doesn’t move much, it means remaining bias is coming from non-parallel trends / time-varying confounding.